# ModelCallLimitMiddleware中间件可以限制调用模型的次数，扩展成本

## 举例1，优雅退出
### 设置exit_behavior='end'

In [2]:
from langchain_core import messages
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware, TodoListMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# 1.初始化模型给中间件使用，不是agent
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)
# 2,创建agent
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2, # 每个线程最多2次模型调用
            # run_limit=5, # 每次运行最多5次
            exit_behavior='end' # 达到限制后默默退出，不会报错
        ),
    ]
)

config = {"configurable":{"thread_id":"1"}} # 作用保证下面的代码都在同一个会话中

print("=" * 30, "> first <", "=" * 30)

first_resp = agent.invoke({"messages":[HumanMessage("你好")]},config=config)
for msg in first_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> second <", "=" * 30)
second_resp = agent.invoke({"messages":[HumanMessage("你是谁？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> third <", "=" * 30)
third_resp = agent.invoke({"messages":[HumanMessage("你是能帮我做什么？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！请问有什么我可以帮你的吗？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！请问有什么我可以帮你的吗？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是 Gemma 4，是由 Google DeepMind 开发的大型语言模型。我是一个开放权重的模型，可以协助你处理文本、理解图像，并回答各种问题。请问有什么我可以帮你的吗？
============================== > third < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==========================

## 举例2，抛异常
### 设置exit_behavior='error'

In [3]:
from langchain_core import messages
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware, TodoListMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# 1.初始化模型给中间件使用，不是agent
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)
# 2,创建agent
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2, # 每个线程最多2次模型调用
            # run_limit=5, # 每次运行最多5次
            exit_behavior='error' # 抛异常
        ),
    ]
)

config = {"configurable":{"thread_id":"1"}} # 作用保证下面的代码都在同一个会话中

print("=" * 30, "> first <", "=" * 30)

first_resp = agent.invoke({"messages":[HumanMessage("你好")]},config=config)
for msg in first_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> second <", "=" * 30)
second_resp = agent.invoke({"messages":[HumanMessage("你是谁？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> third <", "=" * 30)
third_resp = agent.invoke({"messages":[HumanMessage("你是能帮我做什么？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！请问有什么我可以帮您的吗？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！请问有什么我可以帮您的吗？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是一个由 Google 开发的大型语言模型。

你可以把我当做一个全能的助手，我可以帮你：
1. **回答问题**：无论是常识、科学知识还是复杂的专业问题。
2. **创作内容**：比如写文章、诗歌、脚本、邮件等。
3. **翻译语言**：支持多种语言的互译。
4. **编写和调试代码**：支持多种编程语言。
5. **提供建议**：比如旅游计划、学习建议或日常生活中的解决办法。
6. **聊天交流**：陪你讨论任何你感兴趣的话题。

有什么我可以帮你的吗？
============================== > third < ==============================


ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)

## 3.单次调用，优雅退出
### 举例3 配置 run_limit

In [6]:
from pydantic import BaseModel, Field
from langchain_core import messages
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware, TodoListMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# 1.初始化模型给
model = ChatOpenAI(
    model="any", # 这个模型必须和fake_deepseek_server返回的模型名称一致
    api_key="sk12345",
    base_url="http://127.0.0.1:8889", # 必须是fake_deepseek_server提供的url
)
# 2,创建agent
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            # thread_limit=2, # 每个线程最多2次模型调用
            run_limit=3, # 每次运行最多3次
            exit_behavior='end' # 抛异常
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]

)

config = {"configurable":{"thread_id":"1"}} # 作用保证下面的代码都在同一个会话中

print("=" * 30, "> first <", "=" * 30)

first_resp = agent.invoke({"messages":[HumanMessage("你好")]},config=config)
for msg in first_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> second <", "=" * 30)
second_resp = agent.invoke({"messages":[HumanMessage("你是谁？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> third <", "=" * 30)
third_resp = agent.invoke({"messages":[HumanMessage("你是能帮我做什么？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()

Deserializing unregistered type __main__.ContactInfo from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'ContactInfo')]


============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: 康师傅
    email: songhongkang@atguigu.cn
    phone: 12345678912
  EventInfo (call_2)
 Call ID: call_2
  Args:
    event_name: 问数项目启动会
    date: 2026-03-27
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================== Ai Message =========

## 举例4.抛异常，
### 和举例3差不多，只是把exit_behavier改为error

In [8]:
from pydantic import BaseModel, Field
from langchain_core import messages
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware, TodoListMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# 1.初始化模型给
model = ChatOpenAI(
    model="any", # 这个模型必须和fake_deepseek_server返回的模型名称一致
    api_key="sk12345",
    base_url="http://127.0.0.1:8889", # 必须是fake_deepseek_server提供的url
)
# 2,创建agent
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            # thread_limit=2, # 每个线程最多2次模型调用
            run_limit=3, # 每次运行最多3次
            exit_behavior='error' # 抛异常
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]

)

config = {"configurable":{"thread_id":"1"}} # 作用保证下面的代码都在同一个会话中

print("=" * 30, "> first <", "=" * 30)

first_resp = agent.invoke({"messages":[HumanMessage("你好")]},config=config)
for msg in first_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> second <", "=" * 30)
second_resp = agent.invoke({"messages":[HumanMessage("你是谁？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()
print("=" * 30, "> third <", "=" * 30)
third_resp = agent.invoke({"messages":[HumanMessage("你是能帮我做什么？")]},config=config)
for msg in second_resp["messages"]:
    msg.pretty_print()

============================== > first < ==============================


ModelCallLimitExceededError: Model call limits exceeded: run limit (3/3)